# Basic Chatbot

Sequential workflow

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
# Define state

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
# Define model

chat_model = ChatOpenAI(    
    base_url="http://localhost:12434/engines/v1",
    api_key="docker", 
    temperature=0.3, 
    model = "hf.co/bartowski/llama-3.2-1b-instruct-gguf")

In [ ]:
def chat_node(state: AgentState):
    # take user query from the messages
    messages = state['messages']

    # call llm
    response = chat_model.invoke(messages)


    # add to result
    return {'messages': [response]}

In [ ]:
# Define graph

graph = StateGraph(AgentState)

# Add node
graph.add_node('chat_node', chat_node)

# Add nodes
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

# Compile
checkpointer = MemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
# visualize graph
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
# initial_state = {
#     'messages': [HumanMessage(content="What is the capital of India?")]
# }

# final_state = workflow.invoke(initial_state)
# final_state



In [ ]:
# initial_state = {
#     'messages': [HumanMessage(content=" ")]
# }

# final_state = workflow.invoke(initial_state)
# final_state

In [ ]:
thread_id = '1'

while True:
    user_message = input('Type here: ')
    print('User: ', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = workflow.invoke({
        'messages': [HumanMessage(content=user_message)]
    },config=config)

    print('AI: ', response['messages'][-1].content)

In [ ]:
workflow.get_state(config = config)